# Model Training with TensorFlow/Keras

This notebook:
1. Loads prepared data from parquet
2. Splits train/test, fits `StandardScaler` on the training features only (then transforms test), and saves a `LabelEncoder` with Iris class names for decoding predictions
3. Trains a Keras classifier and evaluates on a held-out test split
4. Saves the Keras model plus `scaler.joblib`, and exports SavedModel / ONNX / TFLite

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
import onnx
import os
import joblib

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

# Config

In [ ]:
from config import (
    FEATURE_COLS,
    KERAS_TRAINING_LABEL_ENCODING_PATH,
    KERAS_TRAINING_INPUT_PATH,
    KERAS_TRAINING_DIR,
    KERAS_TRAINING_ACCURACY_AND_LOSS_CHART_PATH,
    KERAS_TRAINING_NATIVE_KERAS_FILE_PATH,
    KERAS_TRAINING_SAVEDMODEL_FILE_PATH,
    KERAS_TRAINING_ONNX_FILE_PATH,
    KERAS_TRAINING_TFLITE_FILE_PATH,
    KERAS_TRAINING_SCALER_JOBLIB_PATH,
)

## Load Data from Parquet

In [ ]:
input_path = KERAS_TRAINING_INPUT_PATH

df = pd.read_parquet(input_path)

In [ ]:
print(f"Data shape: {df.shape}")

In [ ]:
df.head()

In [ ]:
df.info()

## Prepare features and labels

In [ ]:
label_encoding_path=KERAS_TRAINING_LABEL_ENCODING_PATH

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = FEATURE_COLS
label_col = "species_label"

X = df[feature_cols].values.astype(np.float32)
y = df[label_col].values.astype(np.int32)

# LabelEncoder from data prep (same mapping as species_label)
label_encoder = joblib.load(label_encoding_path)
NB_CLASSES = len(label_encoder.classes_)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)


print(f"Raw features shape: {X.shape}")
print(f"Train / test (scaled): {X_train.shape}, {X_test.shape}")
print(f"Labels: train {y_train.shape}, test {y_test.shape}  classes: {label_encoder.classes_}")

## Build the Neural Network Model

In [ ]:
#Number of classes in the target variable

#Create a sequencial model in Keras
model = tf.keras.models.Sequential()

#Add the first hidden layer
model.add(keras.layers.Dense(128,                    #Number of nodes
                             input_shape=(len(feature_cols),),       #Number of input variables
                              name='Hidden-Layer-1', #Logical name
                              activation='relu'))    #activation function

#Add a second hidden layer
model.add(keras.layers.Dense(128,
                              name='Hidden-Layer-2',
                              activation='relu'))

#Add an output layer with softmax activation
model.add(keras.layers.Dense(NB_CLASSES,
                             name='Output-Layer',
                             activation='softmax'))

#Compile the model with loss & metrics (integer class indices -> sparse_categorical_crossentropy)
model.compile(loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

#Print the model meta-data
model.summary()


## Train the Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1,
)

print(f"Final Loss: {history.history['loss'][-1]:.6f}")
print(f"Final Accuracy: {history.history['accuracy'][-1]:.6f}")

## Evaluation

In [ ]:

print("\nAccuracy during Training :\n------------------------------------")
import matplotlib.pyplot as plt

#Plot accuracy of the model after each epoch.
pd.DataFrame(history.history)["accuracy"].plot(figsize=(8, 5))
plt.title("Accuracy improvements with Epoch")
plt.show()

#Evaluate the model against the test dataset and print results
print("\nEvaluation against Test Dataset :\n------------------------------------")
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.6f}, test accuracy: {test_accuracy:.6f}")

## Save Model as TensorFlow SavedModel

### Prepare Models directory

In [ ]:
models_dir = KERAS_TRAINING_DIR

os.makedirs(models_dir, exist_ok=True)

joblib.dump(scaler, KERAS_TRAINING_SCALER_JOBLIB_PATH)
print(f"StandardScaler saved to: {KERAS_TRAINING_SCALER_JOBLIB_PATH}")


### Accuracy and Loss

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot Loss
ax1.plot(history.history['loss'], label='Loss', color='red')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot Accuracy
ax2.plot(history.history['accuracy'], label='Accuracy', color='blue')
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim([0, 1.1])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Save chart to PNG file
chart_path = KERAS_TRAINING_ACCURACY_AND_LOSS_CHART_PATH

plt.savefig(chart_path, dpi=150, bbox_inches='tight')
print(f"Chart saved to: {chart_path}")

plt.show()

### Save in Native Keras Format

In [ ]:
keras_path = KERAS_TRAINING_NATIVE_KERAS_FILE_PATH

# Also save in native Keras format
model.save(keras_path)
print(f"Keras model saved to: {keras_path}")

### Save as TensorFlow SavedModel

In [ ]:
savedmodel_path = KERAS_TRAINING_SAVEDMODEL_FILE_PATH

model.export(savedmodel_path, format='tf_saved_model')
print(f"TensorFlow SavedModel saved to: {savedmodel_path}")

### Export Model to ONNX Format

In [ ]:
onnx_path = KERAS_TRAINING_ONNX_FILE_PATH

# Convert to ONNX format using Keras export
# Export to ONNX using Keras 3's built-in export
model.export(onnx_path, format='onnx')
print(f"ONNX model saved to: {onnx_path}")

### Export Model to TFLite (Mobile) Format

In [ ]:
tflite_path = KERAS_TRAINING_TFLITE_FILE_PATH

# Convert to TFLite format

# Create TFLite converter from Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Convert the model
tflite_model = converter.convert()

# Save TFLite model
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model saved to: {tflite_path}")
print(f"TFLite model size: {len(tflite_model):,} bytes")

### TensorRT (GPU serving)
TensorRT Requirements:

- Requires NVIDIA GPU with CUDA support
- Needs tensorrt package installed (typically via NVIDIA's repositories)
`pip install tensorrt`

## Verify Saved Models

In [ ]:
# List saved models

models_dir = KERAS_TRAINING_DIR
print("Saved model files:")
for root, dirs, files in os.walk(models_dir):
    level = root.replace(models_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"{subindent}{file} ({size:,} bytes)")

## Verify ONNX Model

In [ ]:
# Verify ONNX model
onnx_path = KERAS_TRAINING_ONNX_FILE_PATH

onnx_model_loaded = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model_loaded)
print("✓ ONNX model is valid!")

# Print ONNX model info
print("\nONNX Model Info:")
print(f"  IR Version: {onnx_model_loaded.ir_version}")
print(f"  Opset Version: {onnx_model_loaded.opset_import[0].version}")
print(f"  Producer: {onnx_model_loaded.producer_name}")

print("\nInputs:")
for inp in onnx_model_loaded.graph.input:
    dims = [dim.dim_value if dim.dim_value else 'batch' for dim in inp.type.tensor_type.shape.dim]
    print(f"  - {inp.name}: {dims}")

print("\nOutputs:")
for output in onnx_model_loaded.graph.output:
    dims = [dim.dim_value if dim.dim_value else 'batch' for dim in output.type.tensor_type.shape.dim]
    print(f"  - {output.name}: {dims}")


## Model Export Summary

In [ ]:
print(f"✓ StandardScaler (joblib): {KERAS_SCALER_JOBLIB_PATH}")
print(f"✓ LabelEncoder (joblib): {KERAS_LABEL_ENCODER_JOBLIB_PATH}")
print(f"✓ TensorFlow SavedModel: {savedmodel_path}")
print(f"✓ Keras Model (.keras): {keras_path}")
print(f"✓ ONNX Model: {onnx_path}")
print(f"✓ TFLite Model (mobile): {tflite_path}")